# LOB RL Training on Colab GPU

Train PPO agent for optimal execution in limit order book environment.

**Your data:** 28 CSV files in Google Drive `/csv` folder (512 MB)

**This notebook:**
- ✅ Automatically mounts your Google Drive
- ✅ Auto-splits data 80/20 (train/test)
- ✅ Trains on GPU (T4)
- ✅ Saves models to Drive (persistent)

In [18]:
# Cell 1: Check GPU availability
# Should show: Tesla T4 or similar

!nvidia-smi

Sun Dec 28 20:09:20 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Cell 1.5: Check Colab environment versions
import sys
import subprocess

print("="*60)
print("COLAB ENVIRONMENT CHECK")
print("="*60)

print(f"\nPython version: {sys.version}")
print(f"Python executable: {sys.executable}")

# Check CMake version
try:
    cmake_version = subprocess.run(['cmake', '--version'], capture_output=True, text=True)
    print(f"\nCMake: {cmake_version.stdout.split()[2] if cmake_version.returncode == 0 else 'NOT INSTALLED'}")
except FileNotFoundError:
    print("\nCMake: NOT INSTALLED")

# Check g++ version
try:
    gcc_version = subprocess.run(['g++', '--version'], capture_output=True, text=True)
    print(f"\nG++ version:\n{gcc_version.stdout.split(chr(10))[0]}")
except FileNotFoundError:
    print("\nG++: NOT INSTALLED")

print("\n" + "="*60)
print("⚠️  If Python is 3.12+, you may need to install python3.10-dev instead")
print("="*60)

In [20]:
# Cell 2: Clone your repository
# Replace YOUR_USERNAME with your GitHub username

!git clone https://github.com/ymariam1/lob-sim-orderbook
%cd lob-sim-orderbook

fatal: destination path 'lob-sim-orderbook' already exists and is not an empty directory.
/content/lob-sim-orderbook/lob-sim-orderbook/lob-sim-orderbook


In [22]:
# Cell 3: Install dependencies and build (~3-5 minutes)

print("Step 1: Installing build tools...")
!apt-get update -qq
!apt-get install -y -qq build-essential cmake g++ python3.10-dev
!update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1

print("\nStep 2: Installing pybind11...")
!pip install "pybind11[global]"

print("\nStep 3: Testing CMake directly to see actual errors...")
print("="*60)

# Run CMake manually to see the actual configuration errors
import os
import subprocess

os.makedirs('/tmp/test_build', exist_ok=True)

# Get pybind11 path
import pybind11
pybind11_dir = pybind11.get_cmake_dir()

print(f"pybind11 CMake dir: {pybind11_dir}")
print(f"Python executable: {subprocess.run(['which', 'python3'], capture_output=True, text=True).stdout.strip()}")

print("\nRunning CMake configuration...")
result = subprocess.run(
    [
        'cmake',
        '/content/lob-sim-orderbook/src/cpp',
        f'-DCMAKE_LIBRARY_OUTPUT_DIRECTORY=/tmp/test_build',
        f'-DPYTHON_EXECUTABLE={subprocess.run(["which", "python3"], capture_output=True, text=True).stdout.strip()}',
        '-DCMAKE_BUILD_TYPE=Release',
        f'-Dpybind11_DIR={pybind11_dir}'
    ],
    cwd='/tmp/test_build',
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result.stdout)
print("\nSTDERR:")
print(result.stderr)

if result.returncode != 0:
    print("\n" + "="*60)
    print("❌ CMAKE CONFIGURATION FAILED")
    print("="*60)
    print("This is the actual error we need to fix!")
    print("Look for lines starting with 'CMake Error' or 'Could NOT find'")
else:
    print("\n✅ CMake configuration succeeded!")
    print("Now trying full pip install...")
    !pip install -e . -v

Step 1: Installing build tools...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

Step 2: Installing pybind11...

Step 3: Testing CMake directly to see actual errors...
pybind11 CMake dir: /usr/local/lib/python3.12/dist-packages/pybind11/share/cmake/pybind11
Python executable: /usr/bin/python3

Running CMake configuration...
STDOUT:
-- Found pybind11: /usr/local/lib/python3.12/dist-packages/pybind11/include (found version "3.0.1")
-- Configuring done (0.2s)
-- Generating done (0.0s)
-- Build files have been written to: /tmp/test_build


STDERR:


✅ CMake configuration succeeded!
Now trying full pip install...
Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
Obtaining file:///content/lob-sim-orderbook/lob-sim-orderbook/lob-sim-orderbook
  Running command pip subprocess to install build dependencies
  Using pip 24.1.2

In [23]:
# DIAGNOSTIC CELL - Run this if build fails
# This shows the full error output

print("Running diagnostic build with full output...")
print("="*60)

# Try to build and capture all output
import subprocess
result = subprocess.run(
    ['pip', 'install', '-e', '.', '--verbose'],
    cwd='/content/lob-sim-orderbook',
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result.stdout)
print("\nSTDERR:")
print(result.stderr)
print("="*60)

if result.returncode != 0:
    print(f"\n❌ Build failed with return code: {result.returncode}")
    print("\n📋 Look for the actual error above (usually near the end)")
    print("   Common issues:")
    print("   - Missing CMake")
    print("   - C++ compilation errors")
    print("   - pybind11 not found")
else:
    print("\n✅ Build succeeded!")

Running diagnostic build with full output...
STDOUT:
Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
Obtaining file:///content/lob-sim-orderbook
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Link requires a different Python (3.12.12 not in: '>=3.7,<3.11'): https://files.pythonhosted.org/packages/3a/be/650f9c091ef71cb01d735775d554e068752d3ff63d7943b26316dc401749/numpy-1.21.2.zip (from https://pypi.org/simple/numpy/) (requires-python:>=3.7,<3.11)
  Link requires a different Python (3.12.12 not in: '>=3.7,<3.11'):

In [ ]:
# FALLBACK CELL - Only run if Cell 3 fails with "-march=native" error
# This patches the CMakeLists.txt to work on Colab

print("Applying Colab compatibility patch...")

cmake_file = '/content/lob-sim-orderbook/src/cpp/CMakeLists.txt'

# Read current content
with open(cmake_file, 'r') as f:
    content = f.read()

# Replace -march=native with -march=x86-64 (compatible with Colab)
content = content.replace('-march=native', '-march=x86-64')

# Write back
with open(cmake_file, 'w') as f:
    f.write(content)

print("✅ Patched CMakeLists.txt for Colab compatibility")
print("Now re-run Cell 3 to build with the patch")

# Show the change
!grep -n "march=" /content/lob-sim-orderbook/src/cpp/CMakeLists.txt

## ⚠️ Expected Warnings

You may see CUDA/TensorFlow warnings like:
- `Unable to register cuFFT factory`
- `Unable to register cuDNN factory`
- `computation placer already registered`

**These are normal in Colab and can be safely ignored.** They don't affect training.

In [ ]:
# Cell 4: Mount Google Drive and access your CSV data
# Click the authorization link and allow access

from google.colab import drive
drive.mount('/content/drive')

# Create symlink to your data folder
!mkdir -p data
!ln -sf /content/drive/MyDrive/csv data/csv

# Verify data is accessible
print("First 10 files in your Drive /csv folder:")
!ls -lh data/csv/ | head -10

from pathlib import Path
n_csv = len(list(Path('data/csv').glob('*.csv')))
print(f"\n✅ Found {n_csv} CSV files")

In [ ]:
# Cell 5: Create 80/20 train/test split AUTOMATICALLY
# Works with ANY number of CSV files!

import os
from pathlib import Path

# Get all CSV files (sorted by name)
csv_files = sorted(list(Path('data/csv').glob('*.csv')))
n_files = len(csv_files)

# Calculate 80/20 split (rounded down)
n_train = int(n_files * 0.8)
n_test = n_files - n_train

print("=" * 60)
print("AUTOMATIC TRAIN/TEST SPLIT")
print("=" * 60)
print(f"Total CSV files: {n_files}")
print(f"Train files (80%): {n_train}")
print(f"Test files (20%): {n_test}")
print()

# Create directories
!mkdir -p data/train data/test

# Create symlinks for train set (first 80%)
print("Creating train set...")
for f in csv_files[:n_train]:
    target = Path(f'data/train/{f.name}')
    if not target.exists():
        os.symlink(f, target)

# Create symlinks for test set (remaining 20%)
print("Creating test set...")
for f in csv_files[n_train:]:
    target = Path(f'data/test/{f.name}')
    if not target.exists():
        os.symlink(f, target)

# Verify
train_count = len(list(Path('data/train').glob('*.csv')))
test_count = len(list(Path('data/test').glob('*.csv')))

print(f"\n✅ Train set: {train_count} files")
print(f"✅ Test set: {test_count} files")
print("\nTrain files (older data):")
!ls data/train/ | head -5
print("\nTest files (newer data):")
!ls data/test/

# Set variables for training cell
TRAIN_DATA = "data/train"
TEST_DATA = "data/test"

print(f"\n✅ Ready to train on {n_train} files, test on {n_test} files")

In [ ]:
# Cell 6: Optional - Monitor training with TensorBoard
# Run this cell BEFORE starting training, keep it running

%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/lob_logs

In [ ]:
# Cell 7: Train the model!
# Configuration guide:
#   Quick test (10 mins):  --timesteps 50000  --net-arch 32 32
#   Small (20 mins):       --timesteps 100000 --net-arch 64 64
#   Medium (1 hour):       --timesteps 500000 --net-arch 128 64
#   Large (3 hours):       --timesteps 1000000 --net-arch 128 128 64

!python src/py/train_rl.py \
    --train-data data/train \
    --test-data data/test \
    --timesteps 100000 \
    --target-qty 100 \
    --net-arch 64 64 \
    --batch-size 64 \
    --n-steps 2048 \
    --eval-freq 10000 \
    --save-dir /content/drive/MyDrive/lob_models \
    --log-dir /content/drive/MyDrive/lob_logs

print("\n" + "="*60)
print("✅ TRAINING COMPLETE!")
print("="*60)
print("Models saved to: /content/drive/MyDrive/lob_models/")
print("Logs saved to: /content/drive/MyDrive/lob_logs/")
print("\nYou can access these from your Google Drive anytime!")

In [ ]:
# Cell 8: View and download your trained models

print("Your saved models:")
!ls -lh /content/drive/MyDrive/lob_models/

print("\n" + "="*60)
print("OPTION 1: Access from Google Drive (recommended)")
print("="*60)
print("Go to drive.google.com")
print("Navigate to: lob_models/")
print("Download the folder to your computer")

print("\n" + "="*60)
print("OPTION 2: Download directly from Colab")
print("="*60)
print("Uncomment and run the code below:")
print()

# Uncomment to download:
# from google.colab import files
# !cd /content/drive/MyDrive && zip -r lob_models.zip lob_models/
# files.download('/content/drive/MyDrive/lob_models.zip')

In [ ]:
# Cell 9: Evaluate your trained model

!python src/py/train_rl.py \
    --eval-only \
    --model /content/drive/MyDrive/lob_models/best/best_model \
    --train-data data/test \
    --n-eval-episodes 20 \
    --target-qty 100

In [ ]:
# Cell 10: Compare with baselines (TWAP, VWAP, POV)

# Pick one of your test files
!python src/py/baselines.py \
    --data data/test/blockchain_l3_2025-01-01.csv \
    --strategy both \
    --qty 100